https://habr.com/ru/articles/976510/

<img src="https://habrastorage.org/getpro/habr/upload_files/31c/760/538/31c76053813e5c470e45a5d85fc86ba0.png" width=500>

2E3GX3C3U93E4JK545I35M69L6Q6N78F91PAF9D0CED0GEAF91GQHZRJ6K95LON1TN4QB2S0TMVVR

In [9]:
!pip install mido

In [10]:
from mido import MidiFile, MidiTrack, Message, MetaMessage

def dec_to_base7(n):
    if n == 0:
        return '0'
    digits = []
    while n > 0:
        digits.append(str(n % 7))
        n //= 7
    return ''.join(reversed(digits))


# Исходная строка
s = "2E3GX3C3U93E4JK545I35M69L6Q6N78F91PAF9D0CED0GEAF91GQHZRJ6K95LON1TN4QB2S0TMVVR"
n = int(s, 36)
base7_str = dec_to_base7(n)

# Silicon Valley style: минорная, «холодная» палитра
# 0 = пауза
note_map = {
    '0': None,
    '1': 36,  # C2
    '2': 39,  # Eb2
    '3': 41,  # F2
    '4': 43,  # G2
    '5': 46,  # Bb2
    '6': 48   # C3
}

mid = MidiFile(ticks_per_beat=480)
track = MidiTrack()
mid.tracks.append(track)

# Медленный, вязкий темп
track.append(MetaMessage('set_tempo', tempo=720000, time=0))  # ~83 BPM
track.append(MetaMessage('time_signature', numerator=4, denominator=4, time=0))

time_accumulator = 0

for d in base7_str:
    note = note_map[d]

    if note is None:
        # Пауза = микро-сдвиг ритма
        time_accumulator += 240  # половина бита
    else:
        track.append(Message(
            'note_on',
            note=note,
            velocity=70,
            time=time_accumulator
        ))
        track.append(Message(
            'note_off',
            note=note,
            velocity=0,
            time=240
        ))
        time_accumulator = 0

track.append(MetaMessage('end_of_track', time=0))
mid.save('silicon_valley_style.mid')

print(f"Сгенерировано {len(base7_str)} шагов (вся последовательность)")


Сгенерировано 141 шагов (вся последовательность)
